# 第9周 · 练习：用诊断配方排雷

先用两道体检热身（初始损失、过拟合小 batch），然后进入主戏：**4 段训练代码，各埋一个雷**。它们都能顺利跑完、不报错——你只能从症状推断问题，就像真实研究里一样。最后一道题练习"看曲线下诊断"。

所有代码都在 FashionMNIST 的前 2000 张图上训练 784-256-10 的 MLP（Adam 1e-3、3 epochs、batch 64）。**没有雷的正确版本，测试准确率在 0.77–0.81 之间**——这是你判断"修好了没有"的参照。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from zero2researcher import data, experiments
from zero2researcher.checks import week09 as ck
from zero2researcher.checks import report

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_dataset(cls, train):
    # 下载并缓存（FashionMNIST 约 30MB，只有第一次要等）；网络抖动时自动重试一次
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


fm_train = load_dataset(datasets.FashionMNIST, train=True)
fm_test = load_dataset(datasets.FashionMNIST, train=False)
pixels = fm_train.data.float() / 255.0
MEAN, STD = pixels.mean().item(), pixels.std().item()      # 第 8 周的"魔法数字"算法


def to_inputs(images):
    # uint8 图像 (N, 28, 28) → 标准化并展平的 float 张量 (N, 784)：一次算好，训练时不再逐张变换
    return ((images.float() / 255.0 - MEAN) / STD).reshape(len(images), -1)


X_val, y_val = to_inputs(fm_train.data[50000:52000]), fm_train.targets[50000:52000]   # 固定验证集
X_test, y_test = to_inputs(fm_test.data), fm_test.targets                               # 测试集全量
print(f"mean = {MEAN:.4f}, std = {STD:.4f} | 验证 {len(X_val)} 张 | 测试 {len(X_test)} 张")

X_train, y_train = to_inputs(fm_train.data[:2000]), fm_train.targets[:2000]


def make_mlp(hidden=256):
    return nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10))


def test_accuracy(model):
    model.eval()
    with torch.no_grad():
        return (model(X_test.to(device)).argmax(dim=1).cpu() == y_test).float().mean().item()

## E1 · 体检 ①：初始损失

一位初学者搭了一个两隐层的网络（784-128-128-10）。在训练之前，用训练集的前 256 张图算它的初始损失 `init_loss`，再把**最后一层**权重乘 0.01、bias 清零，重新算一次 `scaled_loss`。两个数分别应该是多少？

In [ ]:
experiments.set_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(device)
criterion = nn.CrossEntropyLoss()
xb, yb = X_train[:256].to(device), y_train[:256].to(device)

# 在这里写你的代码
raise NotImplementedError
print(f"初始损失 {init_loss:.4f} | 末层 ×0.01 之后 {scaled_loss:.4f} | ln(10) = {np.log(10):.4f}")
ck.init_loss(init_loss)
report("末层缩小后的初始损失", abs(scaled_loss - np.log(10)) < 0.01,
       f"应非常接近 ln(10) = 2.3026，实际 {scaled_loss:.4f}",
       "在 torch.no_grad() 下：model[-1].weight.mul_(0.01)；model[-1].bias.zero_()")

## E2 · 体检 ②：过拟合一个小 batch

用训练集的前 **4** 张图、Adam（lr=1e-3）训练一个全新的 784-256-10，训到损失低于 0.05。至少需要多少步？把最后的损失存进 `final_loss`。

**追问**（写一句话）：如果无论训多少步，损失都停在 1.5 附近不动，你会先怀疑什么？

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
x4, y4 = X_train[:4].to(device), y_train[:4].to(device)

# 在这里写你的代码
raise NotImplementedError
print(f"最终损失 {final_loss:.6f}")
ck.overfit_one_batch(final_loss)

（在这里回答追问）

## E3 · 埋雷排查：4 段训练代码

下面每一段都是一位初学者"跑通了"的训练代码，各自埋着**一个** bug。对每一段：

1. 运行它，**记下症状**：每个 epoch 的损失怎么变？测试准确率多少？和"正确版本 0.77–0.81、最后一个 epoch 损失约 0.47"比差在哪？
2. 对照诊断配方（初始损失、过拟合小 batch、损失函数输入、优化器设置）推断 bug 在哪；
3. 在下一格写出修复版，用 `ck` 检查。

**限时 40 分钟**。不要先去找"哪一行和课件不一样"——先读症状，再找代码，这才是真实调试的顺序。

### 雷 a · "先把数据打乱一下"

初学者的说明：*"我担心数据按某种顺序存放，所以训练前把样本打乱了一下。损失降得很慢，是不是模型太小了？"*

In [ ]:
experiments.set_seed(0)
# 打乱样本顺序，防止数据按某种规律存放
perm = torch.randperm(len(X_train))
X = X_train[perm]
y = y_train[torch.randperm(len(y_train))]

model = make_mlp().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
losses = []
for epoch in range(3):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    losses.append(total / n)
print("每个 epoch 的平均训练损失:", np.round(losses, 3), "| 测试准确率:", round(test_accuracy(model), 4))

**症状提问**：损失停在哪个数附近？这个数你在诊断配方里见过吗？测试准确率和"闭眼瞎猜"相比如何？由此推断：模型学到的"输入 → 标签"关系出了什么问题？

（在这里写症状与推断）

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("修复后每个 epoch 的损失:", np.round(losses_a, 3), "| 测试准确率:", round(acc_a, 4))
ck.bug_a_fixed(acc_a)

### 雷 b · "loss 忽高忽低"

初学者的说明：*"第一个 epoch 损失降得挺好，后面反而涨回去了，准确率只有一半左右。是不是 Adam 不稳定，要换 SGD？"*

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
losses = []
for epoch in range(3):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    losses.append(total / n)
print("每个 epoch 的平均训练损失:", np.round(losses, 3), "| 测试准确率:", round(test_accuracy(model), 4))

**症状提问**：损失的走势和正确版本（一路下降到 0.47 左右）有什么不同？"先降后升、来回震荡"提示更新的方向或大小随时间出了问题——在五步核心里，哪一步负责让每次更新只用**当前 batch** 的梯度？

（在这里写症状与推断）

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("修复后每个 epoch 的损失:", np.round(losses_b, 3), "| 测试准确率:", round(acc_b, 4))
ck.bug_b_fixed(acc_b)

### 雷 c · "输出转成概率更好解释"

初学者的说明：*"我在模型输出后面加了 softmax，这样输出就是概率，更好解释。准确率 0.75 左右，比你们低一点点，应该是正常波动吧？"*

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
losses = []
for epoch in range(3):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        probs = torch.softmax(model(xb), dim=1)     # 输出转成概率，更好解释
        loss = criterion(probs, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    losses.append(total / n)
print("每个 epoch 的平均训练损失:", np.round(losses, 3), "| 测试准确率:", round(test_accuracy(model), 4))

**症状提问**：准确率看起来"差不多"，但损失呢？正确版本最后一个 epoch 损失约 0.47，这里停在多少？损失好像撞到了一块**地板**下不去——想一想：如果输入 `CrossEntropyLoss` 的 10 个数都被压在 0 到 1 之间，它算出的"概率"最大能到多少？损失最小能到多少？（提示：$e^1 / (e^1 + 9e^0)$）

（在这里写症状与推断）

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("修复后每个 epoch 的损失:", np.round(losses_c, 3), "| 测试准确率:", round(acc_c, 4))
ck.bug_c_fixed(acc_c, losses_c[-1])

### 雷 d · "调大一点收敛更快"

初学者的说明：*"1e-3 太慢了，我把学习率调大了。损失数字很大，但比第一个 epoch 小多了，是不是再多训几个 epoch 就好了？"*

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e1)     # 1e-3 太慢，调大一点
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
losses = []
for epoch in range(3):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    losses.append(total / n)
print("每个 epoch 的平均训练损失:", np.round(losses, 1), "| 测试准确率:", round(test_accuracy(model), 4))

**症状提问**：损失的**量级**是多少？一个 10 类分类器在"完全瞎猜"时损失才约 2.3——比瞎猜还差几千倍意味着什么？回想 S1 里 SGD 在峡谷里来回横跳的样子。`1e1` 等于多少？

（在这里写症状与推断）

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("修复后每个 epoch 的损失:", np.round(losses_d, 3), "| 测试准确率:", round(acc_d, 4))
ck.bug_d_fixed(acc_d)

**复盘**（两三句话）：四个雷里哪一个最难发现？为什么？如果只允许你在训练前加**一行**检查代码来提前抓住它们中的某一个，你会加什么？

（在这里回答）

## E4 · 看曲线下诊断（选择题）

下面是四次训练的损失曲线（训练集与验证集）。为每一次选一个最合适的诊断：

- **A** 过拟合：加正则化 / 早停 / 更多数据
- **B** 学习率过大：降低学习率
- **C** 模型什么都没学到（损失停在 ln 10）：查数据管道与标签
- **D** 还在进步、尚未收敛：多训几个 epoch 或适当加大模型

In [ ]:
rng = np.random.default_rng(9)
ep = np.arange(1, 31)
noise = lambda s: rng.normal(0, s, ep.size)
runs = {
    "Run 1": (0.9 * np.exp(-ep / 6) + 0.05 + noise(0.01),
              0.55 + 0.35 * np.exp(-ep / 4) + 0.012 * np.clip(ep - 8, 0, None) + noise(0.01)),
    "Run 2": (2.30 - 0.01 * (1 - np.exp(-ep / 5)) + noise(0.005),
              2.30 + noise(0.005)),
    "Run 3": (2.2 * np.exp(-ep / 25) + 0.4 + noise(0.01),
              2.2 * np.exp(-ep / 25) + 0.45 + noise(0.01)),
    "Run 4": (np.abs(3 + 40 * rng.random(ep.size) * (ep > 3)) + noise(0.1),
              np.abs(3 + 40 * rng.random(ep.size) * (ep > 3)) + noise(0.1)),
}
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (name, (tr, va)) in zip(axes, runs.items()):
    ax.plot(ep, tr, label="Train")
    ax.plot(ep, va, label="Validation")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
axes[1].set_ylim(0, 3)      # 固定纵轴：自动缩放会把一条平线放大成"剧烈震荡"
axes[0].legend()
fig.suptitle("Diagnose each run from its loss curves (answer below)")
plt.tight_layout()
plt.show()

**你的答案**（每行一个字母，并用一句话写出你看到的关键证据）：

- Run 1 →
- Run 2 →
- Run 3 →
- Run 4 →